In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM villa_Transformed_Explored"
df = pd.read_sql(query, conn)
df.head()

In [ ]:
# 1. Recréer une colonne "Ville" depuis les colonnes Ville_*
ville_columns = [col for col in df.columns if col.startswith('Ville_')]
df['Ville'] = df[ville_columns].idxmax(axis=1).str.replace("Ville_", "")

# 2. Afficher les quartiers par ville
quartiers_par_ville = df.groupby('Ville')['Quartier'].unique()

# 3. Afficher de façon lisible
for ville, quartiers in quartiers_par_ville.items():
    print(f"\n📍 {ville} :")
    for q in quartiers:
        print(f"  - {q}")


In [ ]:
import warnings

warnings.filterwarnings('ignore')

<h4 style="color:black"> Model 1 - Surface: </h2>

In [ ]:
X1 = df[['Surface_Totale', 'Surface_Habitable']]
y1 = df['Prix']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=5)

model1 =LinearRegression()
model1.fit(X1_train, y1_train)

y1_pred = model1.predict(X1_test)

<h4 style="color:black"> Model 2 - Quartier: </h2>

In [ ]:
X2 = df[['Quartier_enc']]
y2 = df['Prix']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=5)

model2 =LinearRegression()
model2.fit(X2_train, y2_train)

y2_pred = model2.predict(X2_test)

<h4 style="color:black"> Model 3 - Age et Condition: </h2>

In [ ]:
X3 = df[['Age_encoded', 'Standing_encoded', 'Condition_encoded']]
y3 = df['Prix']

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2, random_state=5)

model3 =LinearRegression()
model3.fit(X3_train, y3_train)

y3_pred = model3.predict(X3_test)

<h4 style="color:black"> Model 4 - Room features: </h2>

In [ ]:
X4 = df[['Chambres', 'Salons', 'Salles_De_Bain']]
y4 = df['Prix']

X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.2, random_state=5)

model4 =LinearRegression()
model4.fit(X4_train, y4_train)

y4_pred = model4.predict(X4_test)

<h5 style="color:black"> Accuracy: </h2>

In [ ]:
# Model 1:
r2_1 = r2_score(y1_test, y1_pred)
mse_1 = mean_squared_error(y1_test, y1_pred)
rmse_1 = np.sqrt(mse_1)
y1_test = y1_test.astype(float)
y1_pred = y1_pred.astype(float)
residuals1 = y1_test - y1_pred
SE1 = np.sqrt(np.sum(residuals1**2) / (len(y1_test) - 2))
# Model 2:
r2_2 = r2_score(y2_test, y2_pred)
mse_2 = mean_squared_error(y2_test, y2_pred)
rmse_2 = np.sqrt(mse_2)
y2_test = y2_test.astype(float)
y2_pred = y2_pred.astype(float)
residuals2 = y2_test - y2_pred
SE2 = np.sqrt(np.sum(residuals2**2) / (len(y2_test) - 2))
# Model 3:
r2_3 = r2_score(y3_test, y3_pred)
mse_3 = mean_squared_error(y3_test, y3_pred)
rmse_3 = np.sqrt(mse_3)
y3_test = y3_test.astype(float)
y3_pred = y3_pred.astype(float)
residuals3 = y3_test - y3_pred
SE3 = np.sqrt(np.sum(residuals3**2) / (len(y3_test) - 2))
# Model 4:
r2_4 = r2_score(y4_test, y4_pred)
mse_4 = mean_squared_error(y4_test, y4_pred)
rmse_4 = np.sqrt(mse_4)
y4_test = y4_test.astype(float)
y4_pred = y4_pred.astype(float)
residuals4 = y4_test - y4_pred
SE4 = np.sqrt(np.sum(residuals4**2) / (len(y4_test) - 2))

In [ ]:
print(f'Model 1 R²: {r2_1:.4f}, RMSE: {rmse_1:.2f}, MSE: {mse_1:.2f}, SE: {SE1:.2f}')
print(f'Model 2 R²: {r2_2:.4f}, RMSE: {rmse_2:.2f}, MSE: {mse_2:.2f}, SE: {SE2:.2f}')
print(f'Model 3 R²: {r2_3:.4f}, RMSE: {rmse_3:.2f}, MSE: {mse_3:.2f}, SE: {SE3:.2f}')
print(f'Model 4 R²: {r2_4:.4f}, RMSE: {rmse_4:.2f}, MSE: {mse_4:.2f}, SE: {SE4:.2f}')

<h2 style="color:black"> Effect of different cities </h2>

In [ ]:
villes = ['Ville_Casablanca',
       'Ville_Casablanca_Settat', 'Ville_Fès_Meknès', 'Ville_Marrakech',
       'Ville_Marrakech_Safi', 'Ville_Oriental', 'Ville_Rabat',
       'Ville_Rabat_Sale_Kenitra', 'Ville_Souss_Massa', 'Ville_Tanger',
       'Ville_Tanger_Tetouan_Al_Hoceima']

for ville in villes :
    df[ville] = df[ville].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

In [ ]:
dfs = {}

for ville in villes :
    dfs[ville] = df[df[ville]==1]

In [ ]:
r2_scores = []
mse_score=[]
rmse_score=[]
SE_score=[]

for ville in villes:
    df_ville = dfs[ville].copy()
    X = df_ville.drop(columns=["Prix","Lien","Ville","Age","Standing","Condition","Quartier"]) 
    X = X.dropna(axis=1, how='all')
    imputer = SimpleImputer(strategy='mean')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    y = df_ville["Prix"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44)

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Calculate R² score and othr metrics:
    r2_scores.append(r2_score(y_test, y_pred))
    mse_score.append(mean_squared_error(y_test, y_pred))
    rmse_score.append(np.sqrt(mean_squared_error(y_test, y_pred)))
    y_test = y_test.astype(float)
    y_pred = y_pred.astype(float)
    residuals1 = y_test - y_pred
    SE_score.append( np.sqrt(np.sum(residuals1**2) / (len(y_test) - 2
                                                      )))

In [ ]:
r2_scores_rounded = [round(score, 3) for score in r2_scores]
print(f'r2_score: {r2_scores_rounded}')
print()
mse_score_rounded = [round(score, 3) for score in mse_score]
print(f'mse_score: {mse_score_rounded}')
print()
rmse_score_rounded = [round(score, 3) for score in rmse_score]
print(f'rmse_score: {rmse_score_rounded}')
print()
SE_score_rounded = [round(score, 3) for score in SE_score]
print(f'SE_score: {SE_score_rounded}')

In [ ]:
sns.set_style("whitegrid")
Villes =['Casablanca',
       'Casablanca-Settat', 'Fès-Meknès', 'Marrakech',
       'Marrakech-Safi', 'Oriental', 'Rabat',
       'Rabat-Salé-Kénitra', 'Souss-Massa', 'Tanger',
       'Tanger-Tétouan-Al Hoceima']
plt.figure(figsize=(10,4))
sns.barplot(x=Villes, y=r2_scores, color='darkorange',width = 0.4)
plt.title('R²', fontsize=20)
# plt.xticks(rotation=45)

plt.show()

In [ ]:
results = pd.DataFrame({
    'Ville': Villes,
    'R2': r2_scores
})


In [ ]:
# Conserver uniquement les villes avec un R² positif ou nul
good_results = results[results['R2'] >= 0]

# Liste des bonnes villes
good_villes = good_results['Ville'].tolist()
print("Villes conservées :", good_villes)


In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO FinalDataSet_villa ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()
